# Gesture Painter with AI

In this notebook you can paint on a canvas using your hand. No mouse, no stylus, no touching anything! 

Your index fingertip becomes a brush, tracked in real time by a computer vision gesture detector. Once you've created your sketch, you can pass it (together with a prompt) to an image generation model to finalize your masterpiece!

## Gesture Controls

| Gesture | Action |
|---|---|
| **Pointing Up** (index finger extended) | Draw (your fingertip leaves a trail) |
| **Open Palm** | Clear (wipes the canvas clean) |

### How It Works

Each frame, the app reads the position of your **index fingertip** (`cv.HAND.INDEX_FINGER_TIP`). When the gesture is `Pointing_Up`, it draws a line from the previous fingertip position to the current one.

All line segments are stored in a list and redrawn every frame on top of the live camera feed, so your painting persists as the video updates underneath.

Choose your brush color and size below, and then run the cell.

## Paint!

Set your brush color and size, then run the cell.
To change the brush mid-session: click **Stop** (■), adjust the controls, and run again.

Click **Stop** when you're done and move on to the following cell.

In [ ]:
from codetto import cv
from codetto import graphics
import time

BRUSH_COLOR = "#ff4500" #@param ["#ff4500", "#00bfff", "#00ff88", "#ffff00", "#ffffff", "#ff69b4"]
BRUSH_SIZE  = 6         #@param {type:"slider", min:2, max:20, step:1}

canvas   = graphics.canvas()
camera   = cv.start_camera(canvas, mirror=True)
detector = cv.start_gesture_detector(camera)
ctx      = canvas.get_context('2d')

strokes        = []
last_pos       = None
clear_cooldown = 0

try:
  while True:
    detections = detector.get_detections()
    canvas.draw_hands(detections)

    if detections:
      hand    = detections[0]
      tip     = hand['landmarks'][cv.HAND.INDEX_FINGER_TIP]
      gesture = hand['gesture']

      if clear_cooldown > 0:
        clear_cooldown -= 1
        last_pos = None
      elif gesture == 'Pointing_Up':
        if last_pos:
          strokes.append((last_pos[0], last_pos[1],
                  tip['x'], tip['y'], BRUSH_COLOR, BRUSH_SIZE))
        last_pos = (tip['x'], tip['y'])
      elif gesture == 'Open_Palm':
        strokes.clear()
        ctx.clear_rect(0, 0, 9999, 9999)
        last_pos       = None
        clear_cooldown = 30
      else:
        last_pos = None
    else:
      last_pos = None

    for x1, y1, x2, y2, col, sz in strokes:
      ctx.stroke_style = col
      ctx.line_width   = sz
      ctx.begin_path()
      ctx.move_to(x1, y1)
      ctx.line_to(x2, y2)
      ctx.stroke()

    time.sleep(0.033)
finally:
  detector.stop()
  camera.stop()
  data_url = canvas.to_data_url(include_camera = False)
  assert data_url.startswith('data:image/png;base64,'), data_url[:50]
  print("Camera stopped.")

In [ ]:
#@title Let's extract your sketch...
graphics.display_image(data_url)

In [ ]:
#@title ...and pass your sketch to an image generation model
DESCRIPTION = "Hand drawn sketch of a flower" #@param
STYLE = "Watercolor" #@param ["Anime", "Cartoon", "Watercolor", "Photograph"]
TRANSFORMATION = f"I have a {DESCRIPTION}. Use it to create an image in the style of {STYLE}"

import openai

client = openai.OpenAI(
  base_url='https://openrouter.ai/api/v1',
  api_key=os.environ["OPENROUTER_API_KEY"]
)

print(f"Applying: {TRANSFORMATION}...")

response = client.chat.completions.create(
  model="black-forest-labs/flux.2-klein-4b",
  messages=[{
    "role": "user",
    "content": [
      {"type": "image_url", "image_url": {"url": data_url}},
      {"type": "text", "text": TRANSFORMATION}
    ]
  }],
  extra_body={"modalities": ["image"]}
)

transformed_url = response.choices[0].message.images[0]["image_url"]["url"]
graphics.display_image(transformed_url)

What did you create? What's one thing you'd change or add to make this a more powerful drawing tool?

## Ideas for students to extend this app in the classroom:

- **Color switching** — assign different colors to different gestures (e.g. `Victory` → blue, `Thumb_Up` → yellow)
- **Two-hand mode** — use `num_hands=2` so each hand paints independently
- **Stamps** — detect `Thumb_Up` and paste an image at the fingertip position using `graphics.draw_image()`
- **Undo** — keep track of how many strokes were added per gesture and remove the last group on a specific gesture